# Table 1a: patients per data modality

Produces Supp Table 1a (patients with each modality and with each combination of
modalities, column "corrected") and a per-centre summary of the same table.

Origin: analysis/cohort_desc/sample_number_per_modality.ipynb.

Comparison with the published table: the "as published" column reproduces Supp Table 1a
exactly (WES 274, WES_surv 261, Visium+snRNA+WES_surv 192). Those values counted the clinical
table as WES and did not map the Visium backup sections to their block; with the WES table and
per-patient counts the values are WES 263, WES_surv 250, Visium+snRNA+WES 213, Visium+snRNA+WES_surv 204.

In [ ]:
import pandas as pd
import numpy as np
from gbmcollaboration import AllDataCenters, CHUVDataCenter, UKERDataCenter, CRDataCenter
from gbmcollaboration import (
    SURVIVAL_EXCLUSION
)
from gbmcollaboration.paper import ids, output

OUT = output.setup("table1a_sample_number_per_modality")

def load_clinical_and_wes():
    # -- Loop over the three data centers and merge the clinical data
    merged_df = []
    merged_df_wes = []

    for center in [CHUVDataCenter, UKERDataCenter, CRDataCenter]:
        center_data_dict = center.load_tabular(
            exclude_idh1_mut=True,
            exclude_patient_ids=True,
            exclude_relapse=True
        )
        df = center_data_dict['clinical']['processed gbm clinical']
        df_wes = center_data_dict["wes"]["WES CNV amplification"]


        # To be sure check if we only have Baseline samples
        print(f"---Loading the clinical data from {center.name}---")
        print(df['Sample collection chronology'].value_counts())
        print(f"Found {len(df)} primary cases")
        # append to the merged dataframe
        merged_df.append(df)
        merged_df_wes.append(df_wes)

    # merge the three dataframes
    df_concat = pd.concat(merged_df, ignore_index=False)
    df_concat_wes = pd.concat(merged_df_wes, ignore_index=False)
    print("Dataframes from the three centers have been merged.")

    return df_concat, df_concat_wes

def get_center(idx):
    idx = pd.Index(idx).astype(str)
    return np.select(
        [
            idx.str.startswith("CH"),
            idx.str.startswith("CR"),
            idx.str.startswith("ER"),
        ],
        [
            "CHUV",
            "Charité",
            "UKER",
        ],
        default="Unknown"
    )

def modality_counts(mod_df):
    """Supp Table 1a: patients with each modality and each combination of modalities."""
    mod_counts = mod_df[["Visium", "snRNAseq", "WES", "WES_surv"]].sum()
    joint_mod_subsets = [
        ['Visium','snRNAseq'],
        ['Visium','WES'],
        ['Visium','WES_surv'],
        ['snRNAseq','WES'],
        ['snRNAseq','WES_surv'],
        ['Visium','snRNAseq','WES'],
        ['Visium','snRNAseq','WES_surv']]
    for lst in joint_mod_subsets:
        mod_counts['_'.join(lst)] = mod_df[lst].all(1).sum()
    return mod_counts

In [ ]:
data_sp = AllDataCenters.load_spatial(
    exclude_low_quality_visium = True,
    exclude_idh1_mut = True,
    exclude_patient_ids = True,
    exclude_relapse = True,
    load_deconvolution = True,
    load_gex = False
)
sp_samples = np.hstack([v.celltype_df.index.get_level_values('sample').unique() for k,v in data_sp.items()])
sp_sections = pd.Index(sp_samples)
# section -> block: CH_G_185a_vis and the backup section CH_G_185a_vbu are both CH_G_185a
sp_samples = ids.block_of(sp_sections).unique()
%xdel data_sp

In [ ]:
adata_sc_obs = AllDataCenters.load_singlecell(
    exclude_idh1_mut=True,
    exclude_patient_ids = True,
    exclude_relapse = True,
    obs_only=True,
)
sc_samples = pd.Index(adata_sc_obs['orig.ident'].unique()).astype(str)
print(f'snRNA samples: {len(sc_samples)}')

In [ ]:
df_clin, df_wes = load_clinical_and_wes()
# Exclude CR samples for which survival data is pending
df_wes_surv = df_wes.drop(index=SURVIVAL_EXCLUSION)

clin_samples = df_clin.index.unique()
wes_samples = df_wes.index.unique()
wes_surv_samples = df_wes_surv.index.unique()

In [ ]:
mod_samples = {
    "Visium":sp_samples,
    "snRNAseq":sc_samples,
    "WES":wes_samples,
    "WES_surv":wes_surv_samples,
    "Clinical":clin_samples,
}

mod_patients = {
    k:ids.patient_of(v).unique() for k,v in mod_samples.items()
}

cohort_mod = {
    "samples":mod_samples,
    "patients":mod_patients
}
# Build dataframe with presence/absence
mod_dfs = {}
for lvl, mod_dict in cohort_mod.items():
    union = sorted(set().union(*mod_dict.values()))
    df = pd.DataFrame(index=union)
    for name, lst in mod_dict.items():
        df[name] = df.index.isin(lst)

    mod_dfs[lvl] = df

In [ ]:
# Use patient-level or sample-level table
df = mod_dfs["patients"].copy()      # or mod_dfs["samples"]

df["Center"] = get_center(df.index)

# Count each modality by center
summary = (
    df.groupby("Center")
      .agg({
          "Clinical": "sum",
          "WES_surv": "sum",
          "Visium": "sum",
          "snRNAseq": "sum",
          "WES": "sum",
      })
      .rename(columns={
          "Clinical": "Clinical Data",
          "WES_surv": "WES Survival",
          "Visium": "Spatial Transcriptomics",
          "snRNAseq": "snRNA-seq",
          "WES": "WES",
      })
)

# H&E is available for everyone with Clinical data?
summary.insert(
    summary.columns.get_loc("snRNA-seq"),
    "H&E",
    summary["Clinical Data"]
)

# All modalities
modalities = ["Clinical", "WES_surv", "Visium", "snRNAseq", "WES"]
summary["All Modalities"] = (
    df.groupby("Center")[modalities]
      .apply(lambda x: x.all(axis=1).sum())
)

# Total row
summary.loc["Total"] = summary.sum()

output.save_table(OUT, "patients_per_modality_per_center", summary)
summary

In [ ]:
mod_counts = modality_counts(mod_dfs["patients"])

### As published

The published cell counted `mod_dfs["samples"]` with the clinical table as WES and the
Visium sections only stripped of `_vis`; recomputed here with the same inputs.

In [ ]:
published_samples = {
    "Visium": pd.Index(sp_sections).str.replace('_vis',''),
    "snRNAseq": sc_samples,
    "WES": clin_samples,
    "WES_surv": clin_samples.drop(SURVIVAL_EXCLUSION, errors="ignore"),
}
union = sorted(set().union(*published_samples.values()))
published_df = pd.DataFrame({k: pd.Index(union).isin(v) for k, v in published_samples.items()}, index=union)

table1a = pd.DataFrame({"as published": modality_counts(published_df), "corrected": mod_counts})
output.save_table(OUT, "table1a_patients_per_modality", table1a, panel="Table 1a")
table1a